In [ ]:
# ===========================
# Installation & Imports
# ===========================
# Si tu es dans un notebook, décommente la ligne suivante.
# Si tu es dans un terminal, exécute plutôt:  pip install -q yfinance pandas numpy matplotlib
# !pip install -q yfinance pandas numpy matplotlib

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===========================
# Liste des tickers
# ===========================

us400Tech_tickers = [
"A","AAL","AAP","AAPL","ABBV","ABC","ABNB","ABT","ACGL","ACN",
"ADBE","ADI","ADM","ADP","ADSK","AEE","AEP","AES","AFL","AIG",
"AIZ","AJG","AKAM","ALB","ALGN","ALLE","AMAT","AMCR","AMD","AMGN",
"AMP","AMT","AMZN","ANET","ANSS","AON","AOS","APA","APD","APH",
"APP","APTV","ARE","ARM","ASML","ATVI","AVB","AVGO","AVY","AWK",
"AXON","AXTA","AZO","BA","BAC","BALL","BAX","BBWI","BBY","BDX",
"BEN","BF-B","BIO","BK","BKNG","BKR","BLK","BLL","BMY","BR",
"BRK-B","BRO","BSX","BWA","BX","C","CAG","CAH","CAR","CAT",
"CB","CBOE","CBRE","CCI","CDNS","CDW","CE","CEG","CF","CFG",
"CHD","CHRW","CHTR","CI","CINF","CL","CLX","CMA","CMCSA","CME",
"CMG","CMI","CMS","CNC","COF","COIN","COP","COST","CPRT","CRL",
"CRM","CSCO","CSGP","CSL","CSX","CTAS","CTSH","CTVA","CVX","D",
"DHR","DIS","DLTR","DOV","DOW","DRI","DUK","DVN","DXC","ECL",
"ED","EFX","EIX","EL","ELV","EMN","EMR","ENPH","EOG","EQIX",
"EQR","ES","ESS","ETSY","EW","EXC","EXPD","EXPE","EXR","F",
"FANG","FAST","FDX","FE","FFIV","FICO","FIS","FITB","FLT","FMC",
"FOX","FOXA","FRT","FSLR","FTNT","FTV","GD","GE","GEHC","GEN",
"GILD","GIS","GLW","GM","GNRC","GOOG","GOOGL","GPC","GPN","GRMN",
"GS","GWW","HAL","HAS","HBAN","HCA","HD","HES","HIG","HII",
"HOLX","HON","HOOD","HPE","HPQ","HSIC","HSY","HUM","HWM","IBM",
"ICE","IFF","ILMN","INCY","INTC","INTU","IP","IPG","IQV","IR",
"IRM","ISRG","IT","ITW","IVZ","J","JBHT","JCI","JKHY","JNJ",
"JNPR","JPM","K","KDP","KEY","KEYS","KHC","KIM","KLAC","KMB",
"KMI","KO","KR","L","LDOS","LEN","LH","LIN","LKQ","LLY",
"LMT","LNT","LOW","LRCX","LULU","LYB","MAA","MAR","MAS","MCD",
"MCHP","MCK","MCO","MDT","MET","META","MGM","MHK","MKC","MMM",
"MO","MOS","MPC","MPWR","MRK","MRNA","MS","MSFT","MSI","MU",
"NCLH","NDAQ","NDSN","NEM","NFLX","NI","NKE","NOC","NOW","NRG",
"NSC","NTAP","NUE","NVDA","NVR","NWS","NWSA","O","ODFL","OKE",
"ORCL","ORLY","OTIS","OXY","PANW","PAYX","PCAR","PFE","PG","PGR",
"PH","PHM","PKG","PLD","PLTR","PM","PNC","PPG","PRU","PSA",
"PSX","PTC","PWR","PXD","PYPL","QCOM","RCL","REGN","RF","RIVN",
"ROK","ROL","ROP","ROST","RSG","RTX","SBAC","SBUX","SCHW","SEDG",
"SHW","SIRI","SJM","SLB","SNA","SNPS","SO","SPG","SPGI","SRE",
"STT","STX","STZ","SWK","SWKS","SYK","SYY","T","TGT","TJX",
"TMO","TMUS","TROW","TSCO","TSLA","TSN","TT","TTWO","TXN","TXT",
"UAL","UDR","ULTA","UNH","UNP","UPS","URI","USB","V","VFC",
"VICI","VLO","VMC","VRSK","VRTX","VTR","VTRS","VZ","WAB","WAT",
"WBA","WBD","WDC","WEC","WELL","WFC","WHR","WM","WMB","WMT",
"WRB","WST","WTW","WY","XEL","XOM","XYL","YUM","ZBH","ZBRA"
]

# ===========================
# Téléchargement & Feature Engineering
# ===========================
print("📊 Téléchargement des données financières depuis Yahoo Finance...")
data_list = []

for i, ticker in enumerate(us400Tech_tickers):
    if i % 20 == 0:
        print(f"Progression: {i}/{len(us400Tech_tickers)}")

    try:
        stock = yf.Ticker(ticker)

        # (1) Informations fondamentales (peuvent être partielles selon le ticker)
        info = stock.info  # peut être lent/incomplet pour certains titres
        debt_to_equity = info.get('debtToEquity', np.nan)
        revenue = info.get('totalRevenue', np.nan)

        # (2) Historique des prix (1 an) pour calculer rendement & risque annualisés
        hist = stock.history(period="1y", auto_adjust=True)
        if len(hist) > 0 and 'Close' in hist:
            returns = hist['Close'].pct_change().dropna()
            rate_of_return = returns.mean() * 252.0         # rendement annualisé (moyenne * ~252 jours)
            risk = returns.std(ddof=0) * np.sqrt(252.0)     # volatilité annualisée
        else:
            rate_of_return = np.nan
            risk = np.nan

        data_list.append({
            'Ticker': ticker,
            'Debt_to_Equity': debt_to_equity,
            'Risk': risk,
            'Return': rate_of_return,
            'Revenue': revenue
        })

    except Exception as e:
        # On ignore les tickers problématiques pour garder le script robuste
        continue

df = pd.DataFrame(data_list)
print(f"\n✅ Données récupérées pour {len(df)} entreprises (avec NaN possibles)")
# ===========================
# Télécharger le DataFrame en CSV
# ===========================
df.to_csv('tech_companies_data.csv', index=False)
print("✅ DataFrame téléchargé sous le nom 'tech_companies_data.csv'")

from google.colab import files

files.download('tech_companies_data.csv')

📊 Téléchargement des données financières depuis Yahoo Finance...
Progression: 0/400


ERROR:yfinance:$ABC: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")


Progression: 20/400


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
ERROR:yfinance:$ANSS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")


Progression: 40/400


...

Progression: 380/400


ERROR:yfinance:$WBA: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")



✅ Données récupérées pour 400 entreprises (avec NaN possibles)
✅ DataFrame téléchargé sous le nom 'tech_companies_data.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task
Télécharger un fichier CSV contenant les données des 300 plus grandes entreprises du SP500 après avoir effectué un feature engineering.

## Trouver les tickers du sp500

### Subtask:
Obtenir une liste des tickers des entreprises incluses dans l'indice SP500.


**Reasoning**:
Import necessary libraries and fetch the SP500 components from Wikipedia.



In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', {'class': 'wikitable sortable'})
sp500_tickers = []
for row in table.tbody.find_all('tr')[1:]: # Skip the header row
    ticker = row.find_all('td')[0].text.strip()
    sp500_tickers.append(ticker)

print(f"Found {len(sp500_tickers)} SP500 tickers.")
print(sp500_tickers[:10]) # Display the first 10 tickers

AttributeError: 'NoneType' object has no attribute 'tbody'

**Reasoning**:
The previous command failed because the table was not found. I need to find the correct table in the HTML. I will inspect the HTML structure to locate the table containing the SP500 tickers and modify the table finding logic.



In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all tables and iterate to find the one with 'Symbol' and 'Security' columns
sp500_tickers = []
tables = soup.find_all('table', {'class': 'wikitable'})
for table in tables:
    if table.find('th', string='Symbol') and table.find('th', string='Security'):
        for row in table.find_all('tr')[1:]: # Skip the header row
            ticker = row.find_all('td')[0].text.strip()
            sp500_tickers.append(ticker)
        break # Found the table, exit the loop

if sp500_tickers:
    print(f"Found {len(sp500_tickers)} SP500 tickers.")
    print(sp500_tickers[:10]) # Display the first 10 tickers
else:
    print("Could not find the SP500 tickers table.")

Could not find the SP500 tickers table.


**Reasoning**:
The previous command failed to find the table. I need to inspect the HTML content more carefully to identify the correct table and its class or attributes. I will print the HTML content to examine the table structure and find a reliable way to locate the SP500 list.



In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Print a part of the HTML to inspect the table structure
print(soup.prettify()[:2000])

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also T400119.



**Reasoning**:
The previous command received a warning about setting a user-agent. I need to add a user-agent to the request headers to avoid this issue and try to locate the table again.



In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Try to find the table again using a more specific class or attribute if available
# Based on typical Wikipedia tables for lists, 'wikitable sortable' is a common class.
# Let's retry with this class and also look for the caption or header to confirm it's the right table.
sp500_tickers = []
table = soup.find('table', {'class': 'wikitable sortable'})

if table:
    # Check if the table has the expected headers
    headers = [th.text.strip() for th in table.find_all('th')]
    if 'Symbol' in headers and 'Security' in headers:
        for row in table.find_all('tr')[1:]: # Skip the header row
            ticker = row.find_all('td')[0].text.strip()
            sp500_tickers.append(ticker)

if sp500_tickers:
    print(f"Found {len(sp500_tickers)} SP500 tickers.")
    print(sp500_tickers[:10]) # Display the first 10 tickers
else:
    print("Could not find the SP500 tickers table.")